In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import time

options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

try:
    # Open Etherscan
    driver.get("https://etherscan.io/")
    driver.maximize_window()

    # Wait for the dropdown to be clickable
    wait = WebDriverWait(driver, 10)
    dropdown = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "filterby")))

    # Click on the dropdown menu
    dropdown.click()

    # Wait and select "Tokens" from the dropdown
    token_option = wait.until(EC.element_to_be_clickable((By.XPATH, "//option[@value='2']")))
    token_option.click()

    time.sleep(5)  # Keep browser open for a few seconds

finally:
    driver.quit()


In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# Load the CSV file (Update with your actual file path)
file_path = r"C:\Users\pcagm\Downloads\token_data_march_4.csv"
df = pd.read_csv(file_path)

# Filter for Ethereum tokens where "wasRekt = 1"
rekt_tokens = df[
    (df["contract_chain"].str.contains("ethereum", na=False, case=False)) & 
    (df["wasRekt"] == 1)
][["Symbol", "Name", "contract_address"]].dropna()

# Convert DataFrame to list of tuples (Symbol, Name)
eth_tokens = list(zip(rekt_tokens["Symbol"], rekt_tokens["Name"]))

# Initialize Chrome WebDriver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

# Set of searched tokens to avoid repetition
searched_tokens = set()

# Define storage for extracted data
all_data = []

try:
    # Open Etherscan
    driver.get("https://etherscan.io/")
    driver.maximize_window()

    for symbol, name in eth_tokens:
        if symbol in searched_tokens:
            print(f"Skipping {symbol}, already searched.")
            continue

        # Wait for the search input
        search_input = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "search-panel"))
        )

        # Enter token symbol
        search_input.clear()
        search_input.send_keys(symbol)
        time.sleep(3)

        # Wait for dropdown results
        dropdown_results = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "auto-search-panel-results"))
        )

        # Get all search results
        search_results = dropdown_results.find_elements(By.TAG_NAME, "li")

        best_match = None
        for result in search_results:
            try:
                token_name_element = result.find_element(By.CSS_SELECTOR, "div.text-truncate.me-2")
                if token_name_element and name.lower() in token_name_element.text.lower():
                    best_match = result
                    break
            except:
                continue

        if best_match:
            best_match.click()
            time.sleep(5)
            searched_tokens.add(symbol)

            # Click "Holders" tab
            holders_tab = WebDriverWait(driver, 15).until(
                EC.element_to_be_clickable((By.ID, "ContentPlaceHolder1_tabHolders"))
            )
            holders_tab.click()
            time.sleep(5)

            # Extract data from the table
            while True:
                try:
                    # Wait for table body
                    tbody = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "tbody.align-middle.text-nowrap"))
                    )
                    rows = tbody.find_elements(By.TAG_NAME, "tr")

                    for row in rows:
                        cols = row.find_elements(By.TAG_NAME, "td")
                        if len(cols) >= 5:
                            rank = cols[0].text.strip()
                            address = cols[1].text.strip()
                            quantity = cols[2].text.strip()
                            percentage = cols[3].text.strip()
                            value = cols[4].text.strip()

                            all_data.append([symbol, rank, address, quantity, percentage, value])

                    # Look for "Next Page" button
                    next_page_buttons = driver.find_elements(By.XPATH, "//a[contains(@aria-label, 'Next')]")

                    next_page_button = None
                    for button in next_page_buttons:
                        if "disabled" not in button.get_attribute("class"):
                            next_page_button = button
                            break

                    if next_page_button:
                        next_page_button.click()
                        time.sleep(5)
                    else:
                        break

                except:
                    print(f"No more pages left for {symbol}. Moving to the next token.")
                    break

        else:
            print(f"No exact match found for {symbol}")

        # Navigate back to homepage
        driver.get("https://etherscan.io/")
        time.sleep(2)

finally:
    driver.quit()

    # Convert extracted data to a DataFrame
    columns = ["Symbol", "Rank", "Address", "Quantity", "Percentage", "Value"]
    df_holders = pd.DataFrame(all_data, columns=columns)

    # Save to CSV
    output_file = r"C:\Users\pcagm\Downloads\token_holders_data.csv"
    df_holders.to_csv(output_file, index=False)

    print(f"Data saved to {output_file}")


Skipping 1mil, already searched.
No exact match found for agn
No exact match found for agn
No exact match found for alphr
No exact match found for alphr
No exact match found for amc
No exact match found for amc
No exact match found for atf
No exact match found for atf
No exact match found for apn
No exact match found for apn
Data saved to C:\Users\pcagm\Downloads\token_holders_data.csv


ElementClickInterceptedException: Message: element click intercepted: Element <li class="nav-item nav-link mb-2 rounded" style="cursor: pointer;" role="option" tabindex="-1" aria-selected="false" aria-setsize="30" aria-posinset="7">...</li> is not clickable at point (606, 715). Other element would receive the click: <div class="alert alert-light fade show border shadow-sm d-inline-flex flex-wrap flex-sm-nowrap align-items-sm-center text-start gap-3 mx-3" role="alert">...</div>
  (Session info: chrome=133.0.6943.143)
Stacktrace:
	GetHandleVerifier [0x00007FF6F5A2C6A5+28789]
	(No symbol) [0x00007FF6F5995B20]
	(No symbol) [0x00007FF6F5828F9A]
	(No symbol) [0x00007FF6F58871E9]
	(No symbol) [0x00007FF6F5884BA2]
	(No symbol) [0x00007FF6F5881C51]
	(No symbol) [0x00007FF6F5880B51]
	(No symbol) [0x00007FF6F5872314]
	(No symbol) [0x00007FF6F58A732A]
	(No symbol) [0x00007FF6F5871BC6]
	(No symbol) [0x00007FF6F58A7540]
	(No symbol) [0x00007FF6F58CF7E3]
	(No symbol) [0x00007FF6F58A7103]
	(No symbol)

In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# Load the CSV file (Update with your actual file path)
file_path = r"C:\Users\pcagm\Downloads\Untitled spreadsheet - token_data_march_4.csv"
df = pd.read_csv(file_path)

# Filter for Ethereum tokens where "wasRekt = 1"
rekt_tokens = df[
    (df["contract_chain"].str.contains("ethereum", na=False, case=False)) & 
    (df["wasRekt"] == 1)
][["Symbol", "Name", "contract_address"]].dropna()

# Convert DataFrame to list of tuples (Symbol, Name, Contract Address)
eth_tokens = list(zip(rekt_tokens["Symbol"], rekt_tokens["Name"], rekt_tokens["contract_address"]))

# Initialize Chrome WebDriver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

# Set of searched tokens to avoid repetition
searched_tokens = set()

# Define storage for extracted data
all_data = []

try:
    # Open Etherscan and wait for 5 seconds to fully load
    driver.get("https://etherscan.io/")
    time.sleep(5)  # Ensuring page loads completely
    driver.maximize_window()

    for symbol, name, contract_address in eth_tokens:
        if symbol in searched_tokens:
            print(f"Skipping {symbol}, already searched.")
            continue

        # Search for token in Etherscan search bar
        try:
            # Wait for search input field
            search_input = WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.ID, "search-panel"))
            )

            # Enter token symbol
            search_input.clear()
            search_input.send_keys(symbol)
            time.sleep(3)  # Wait for results to populate

            # Wait for dropdown results
            dropdown_results = WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.ID, "auto-search-panel-results"))
            )

            # Get all search results
            search_results = dropdown_results.find_elements(By.TAG_NAME, "li")

            best_match = None
            for result in search_results:
                if symbol.lower() in result.text.lower():
                    best_match = result
                    break

            if best_match:
                best_match.click()
                time.sleep(5)  # Wait for token page to load
            else:
                print(f"Token {symbol} not found. Skipping...")
                continue

            searched_tokens.add(symbol)

            # Navigate to Token Holders page
            token_holders_url = f"https://etherscan.io/token/{contract_address}#balances"
            driver.get(token_holders_url)
            time.sleep(5)  # Wait for page to load before extracting data

            # Extract data from the table
            while True:
                try:
                    # Wait for table body
                    tbody = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "tbody.align-middle.text-nowrap"))
                    )
                    rows = tbody.find_elements(By.TAG_NAME, "tr")

                    for row in rows:
                        cols = row.find_elements(By.TAG_NAME, "td")
                        if len(cols) >= 6:
                            rank = cols[0].text.strip()
                            
                            # Extract full address (including display name and raw address)
                            address_element = cols[1].find_element(By.CSS_SELECTOR, "a")
                            address_name = address_element.text.strip()
                            full_address = address_element.get_attribute("data-clipboard-text").strip()
                            
                            quantity = cols[2].text.strip()
                            percentage = cols[3].text.strip().split()[0]  # Extract just the percentage value
                            value = cols[4].text.strip()

                            # Extract analytics link
                            analytics_link_element = cols[5].find_element(By.TAG_NAME, "a")
                            analytics_link = analytics_link_element.get_attribute("href")

                            all_data.append([symbol, rank, address_name, full_address, quantity, percentage, value, analytics_link])

                    # Look for "Next Page" button
                    next_page_buttons = driver.find_elements(By.XPATH, "//a[contains(@aria-label, 'Next')]")

                    next_page_button = None
                    for button in next_page_buttons:
                        if "disabled" not in button.get_attribute("class"):
                            next_page_button = button
                            break

                    if next_page_button:
                        next_page_button.click()
                        time.sleep(5)
                    else:
                        break

                except:
                    print(f"No more pages left for {symbol}. Moving to the next token.")
                    break

        except Exception as e:
            print(f"Error searching for {symbol}: {e}")

finally:
    driver.quit()

    # Convert extracted data to a DataFrame
    columns = ["Symbol", "Rank", "Address Name", "Full Address", "Quantity", "Percentage", "Value", "Analytics Link"]
    df_holders = pd.DataFrame(all_data, columns=columns)

    # Save to CSV
    output_file = r"C:\Users\pcagm\Downloads\token_holders_data.csv"
    df_holders.to_csv(output_file, index=False)

    print(f"Data saved to {output_file}")


No more pages left for 1mil. Moving to the next token.
Skipping 1mil, already searched.
Error searching for agn: Message: invalid session id
Stacktrace:
	GetHandleVerifier [0x00007FF6F5A2C6A5+28789]
	(No symbol) [0x00007FF6F5995B20]
	(No symbol) [0x00007FF6F5828DCC]
	(No symbol) [0x00007FF6F586F1CF]
	(No symbol) [0x00007FF6F58A71F2]
	(No symbol) [0x00007FF6F58A1B89]
	(No symbol) [0x00007FF6F58A0C39]
	(No symbol) [0x00007FF6F57F5595]
	GetHandleVerifier [0x00007FF6F5D71AED+3458237]
	GetHandleVerifier [0x00007FF6F5D8829C+3550316]
	GetHandleVerifier [0x00007FF6F5D7DB9D+3507565]
	GetHandleVerifier [0x00007FF6F5AF2C6A+841274]
	(No symbol) [0x00007FF6F59A09EF]
	(No symbol) [0x00007FF6F57F41AE]
	GetHandleVerifier [0x00007FF6F5DF66A8+4001912]
	BaseThreadInitThunk [0x00007FFA79ABE8D7+23]
	RtlUserThreadStart [0x00007FFA7A13BF2C+44]

Error searching for agn: Message: invalid session id
Stacktrace:
	GetHandleVerifier [0x00007FF6F5A2C6A5+28789]
	(No symbol) [0x00007FF6F5995B20]
	(No symbol) [0x00007

In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# Load the CSV file (Update with your actual file path)
file_path = r"C:\Users\pcagm\Downloads\token_data_march_4.csv"
df = pd.read_csv(file_path)

# Filter for Ethereum tokens where "wasRekt = 1"
rekt_tokens = df[
    (df["contract_chain"].str.contains("ethereum", na=False, case=False)) & 
    (df["wasRekt"] == 1)
][["Symbol", "Name", "contract_address"]].dropna()

# Convert DataFrame to list of tuples (Symbol, Name)
eth_tokens = list(zip(rekt_tokens["Symbol"], rekt_tokens["Name"]))

# Initialize Chrome WebDriver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

# Storage for extracted data
all_data = []

try:
    driver.get("https://etherscan.io/")
    driver.maximize_window()
    
    for symbol, name in eth_tokens:
        # Search for the token
        search_input = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "search-panel"))
        )
        search_input.clear()
        search_input.send_keys(symbol)
        time.sleep(3)

        # Wait for dropdown results and select the best match
        dropdown_results = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "auto-search-panel-results"))
        )
        search_results = dropdown_results.find_elements(By.TAG_NAME, "li")
        
        best_match = None
        for result in search_results:
            try:
                token_name_element = result.find_element(By.CSS_SELECTOR, "div.text-truncate.me-2")
                if token_name_element and name.lower() in token_name_element.text.lower():
                    best_match = result
                    break
            except:
                continue
        
        if best_match:
            best_match.click()
            time.sleep(5)
            
            # Click "Holders" tab
            holders_tab = WebDriverWait(driver, 15).until(
                EC.element_to_be_clickable((By.ID, "ContentPlaceHolder1_tabHolders"))
            )
            holders_tab.click()
            time.sleep(5)

            # Extract table headers dynamically
            table_headers = driver.find_elements(By.XPATH, "//thead[@id='theadHolderTable']//th")
            header_names = [header.text.strip() for header in table_headers]
            
            # Extract data from the holders' table
            while True:
                try:
                    tbody = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "tbody.align-middle.text-nowrap"))
                    )
                    rows = tbody.find_elements(By.TAG_NAME, "tr")

                    for row in rows:
                        cols = row.find_elements(By.TAG_NAME, "td")
                        if len(cols) >= 5:
                            rank = cols[0].text.strip()
                            address = cols[1].text.strip()
                            quantity = cols[2].text.strip()
                            percentage = cols[3].text.strip()
                            value = cols[4].text.strip()
                            
                            all_data.append([symbol, rank, address, quantity, percentage, value])

                    # Look for "Next Page" button
                    next_page_buttons = driver.find_elements(By.XPATH, "//a[contains(@aria-label, 'Next')]")
                    next_page_button = None
                    for button in next_page_buttons:
                        if "disabled" not in button.get_attribute("class"):
                            next_page_button = button
                            break
                    
                    if next_page_button:
                        next_page_button.click()
                        time.sleep(5)
                    else:
                        break

                except:
                    print(f"No more pages left for {symbol}. Moving to the next token.")
                    break
        else:
            print(f"No exact match found for {symbol}")

        driver.get("https://etherscan.io/")
        time.sleep(2)

finally:
    driver.quit()
    
    # Convert extracted data to a DataFrame
    columns = ["Symbol", "Rank", "Address", "Quantity", "Percentage", "Value"]
    df_holders = pd.DataFrame(all_data, columns=columns)
    
    # Save to CSV
    output_file = r"C:\Users\pcagm\Downloads\token_holders_data.csv"
    df_holders.to_csv(output_file, index=False)
    
    print(f"Data saved to {output_file}")

No exact match found for agn
No exact match found for agn
No exact match found for alphr
No exact match found for alphr
No exact match found for amc
No exact match found for amc
No exact match found for atf
No exact match found for atf
No exact match found for apn
No exact match found for apn
Data saved to C:\Users\pcagm\Downloads\token_holders_data.csv


ElementClickInterceptedException: Message: element click intercepted: Element <li class="nav-item nav-link mb-2 rounded" style="cursor: pointer;" role="option" tabindex="-1" aria-selected="false" aria-setsize="30" aria-posinset="7">...</li> is not clickable at point (606, 715). Other element would receive the click: <div class="alert alert-light fade show border shadow-sm d-inline-flex flex-wrap flex-sm-nowrap align-items-sm-center text-start gap-3 mx-3" role="alert">...</div>
  (Session info: chrome=133.0.6943.143)
Stacktrace:
	GetHandleVerifier [0x00007FF6F5A2C6A5+28789]
	(No symbol) [0x00007FF6F5995B20]
	(No symbol) [0x00007FF6F5828F9A]
	(No symbol) [0x00007FF6F58871E9]
	(No symbol) [0x00007FF6F5884BA2]
	(No symbol) [0x00007FF6F5881C51]
	(No symbol) [0x00007FF6F5880B51]
	(No symbol) [0x00007FF6F5872314]
	(No symbol) [0x00007FF6F58A732A]
	(No symbol) [0x00007FF6F5871BC6]
	(No symbol) [0x00007FF6F58A7540]
	(No symbol) [0x00007FF6F58CF7E3]
	(No symbol) [0x00007FF6F58A7103]
	(No symbol)

In [35]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# Load the CSV file (Update with your actual file path)
file_path = r"C:\Users\pcagm\Downloads\token_data_march_4.csv"
df = pd.read_csv(file_path)

# Filter for Ethereum tokens where "wasRekt = 1"
rekt_tokens = df[
    (df["contract_chain"].str.contains("ethereum", na=False, case=False)) & 
    (df["wasRekt"] == 1)
][["Symbol", "Name", "contract_address"]].dropna()

# Convert DataFrame to list of tuples (Symbol, Name)
eth_tokens = list(zip(rekt_tokens["Symbol"], rekt_tokens["Name"]))

# Initialize Chrome WebDriver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

# Storage for extracted data
all_data = []
searched_tokens = set()  # Keep track of searched symbols

try:
    driver.get("https://etherscan.io/")
    driver.maximize_window()
    
    for symbol, name in eth_tokens:
        if symbol in searched_tokens:
            print(f"Skipping {symbol}, already searched.")
            continue

        # Search for the token
        search_input = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "search-panel"))
        )
        search_input.clear()
        search_input.send_keys(symbol)
        time.sleep(3)

        # Wait for dropdown results and select the best match
        dropdown_results = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "auto-search-panel-results"))
        )
        search_results = dropdown_results.find_elements(By.TAG_NAME, "li")
        
        best_match = None
        for result in search_results:
            try:
                token_name_element = result.find_element(By.CSS_SELECTOR, "div.text-truncate.me-2")
                if token_name_element and name.lower() in token_name_element.text.lower():
                    best_match = result
                    break
            except:
                continue
        
        if best_match:
            best_match.click()
            time.sleep(5)
            searched_tokens.add(symbol)  # Mark symbol as searched
            
            # Click "Holders" tab
            holders_tab = WebDriverWait(driver, 15).until(
                EC.element_to_be_clickable((By.ID, "ContentPlaceHolder1_tabHolders"))
            )
            holders_tab.click()
            time.sleep(5)

            # Extract table headers dynamically with explicit waiting
            try:
                # Wait for the correct table to load
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.XPATH, "//table[contains(@class, 'table') and contains(@id, 'ContentPlaceHolder1_')]"))
                )
                
                # Now try finding the correct headers within the "Holders" table
                table_headers = driver.find_elements(By.XPATH, "//table[contains(@class, 'table') and contains(@id, 'ContentPlaceHolder1_')]//thead//th")
                
                # Extract header names, filtering out any empty or irrelevant ones
                header_names = [header.text.strip() for header in table_headers if header.text.strip() and "Exchange" not in header.text and "Pair" not in header.text]
                
                if not header_names:
                    print("Warning: No valid headers found. Printing raw header elements for debugging.")
                    for header in table_headers:
                        print(f"Raw header element: {header.get_attribute('outerHTML')}")
                else:
                    print(f"\nExtracted Headers: {header_names}")

            except Exception as e:
                print(f"Error retrieving headers: {e}")
            
            # Extract data from the holders' table
            while True:
                try:
                    tbody = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "tbody.align-middle.text-nowrap"))
                    )
                    rows = tbody.find_elements(By.TAG_NAME, "tr")

                    for row in rows:
                        cols = row.find_elements(By.TAG_NAME, "td")
                        if len(cols) >= 5:
                            rank = cols[0].text.strip()
                            address = cols[1].text.strip()
                            quantity = cols[2].text.strip()
                            percentage = cols[3].text.strip()
                            value = cols[4].text.strip()
                            
                            data_entry = [symbol, rank, address, quantity, percentage, value]
                            print(f"Extracted Data: {data_entry}")  # Print scraped data
                            all_data.append(data_entry)

                    # Look for "Next Page" button
                    next_page_buttons = driver.find_elements(By.XPATH, "//a[contains(@aria-label, 'Next')]")
                    next_page_button = None
                    for button in next_page_buttons:
                        if "disabled" not in button.get_attribute("class"):
                            next_page_button = button
                            break
                    
                    if next_page_button:
                        next_page_button.click()
                        time.sleep(5)
                    else:
                        break

                except:
                    print(f"No more pages left for {symbol}. Moving to the next token.")
                    break
        else:
            print(f"No exact match found for {symbol}")

        driver.get("https://etherscan.io/")
        time.sleep(2)

finally:
    driver.quit()
    
    # Convert extracted data to a DataFrame
    columns = ["Symbol", "Rank", "Address", "Quantity", "Percentage", "Value"]
    df_holders = pd.DataFrame(all_data, columns=columns)
    
    # Save to CSV
    output_file = r"C:\Users\pcagm\Downloads\token_holders_data.csv"
    df_holders.to_csv(output_file, index=False)
    
    print(f"Data saved to {output_file}")


Error retrieving headers: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF6F5A2C6A5+28789]
	(No symbol) [0x00007FF6F5995B20]
	(No symbol) [0x00007FF6F5828F9A]
	(No symbol) [0x00007FF6F587F346]
	(No symbol) [0x00007FF6F587F57C]
	(No symbol) [0x00007FF6F58D2B17]
	(No symbol) [0x00007FF6F58A736F]
	(No symbol) [0x00007FF6F58CF7E3]
	(No symbol) [0x00007FF6F58A7103]
	(No symbol) [0x00007FF6F586FFC0]
	(No symbol) [0x00007FF6F5871273]
	GetHandleVerifier [0x00007FF6F5D71AED+3458237]
	GetHandleVerifier [0x00007FF6F5D8829C+3550316]
	GetHandleVerifier [0x00007FF6F5D7DB9D+3507565]
	GetHandleVerifier [0x00007FF6F5AF2C6A+841274]
	(No symbol) [0x00007FF6F59A09EF]
	(No symbol) [0x00007FF6F599CB34]
	(No symbol) [0x00007FF6F599CCD6]
	(No symbol) [0x00007FF6F598C119]
	BaseThreadInitThunk [0x00007FFA79ABE8D7+23]
	RtlUserThreadStart [0x00007FFA7A13BF2C+44]

Skipping 1mil, already searched.
No exact match found for agn
No exact match found for agn
No exact match found for alphr
No exact match found for al

KeyboardInterrupt: 